In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
N_PEERS = 130
RESULTS_PATH = "results"
N_ITERS = 10
XLIM_LEFT = 69
XLIM_RIGHT = 100

In [ ]:
def parse_ifstat_logs() -> list[list[float]]:
    # returns ifstat sum list per host.
    file_paths = [f'../{RESULTS_PATH}/{N_PEERS}/{seed}/ifstat_h{i+1}.log' for seed in range(N_ITERS) for i in range(N_PEERS)]

    raw_data = []  # This will contain N lists (one per file)
    for file_path in file_paths:
        file_sums = []  # Sum of the two floats per line
        with open(file_path, 'r') as f:
            # Skip first two title lines
            next(f, None)
            next(f, None)

            for line in f:
                parts = line.strip().split()
                
                val1 = float(parts[0])
                val2 = float(parts[1])
                file_sums.append(val1 + val2)
        raw_data.append(file_sums)

    min_length = min(len(sums) for sums in raw_data)
    return [entry[:min_length] for entry in raw_data]

In [ ]:
def main():
    # Plot each list as a separate line
    window = 10

    data = parse_ifstat_logs()

    with open(f'../{RESULTS_PATH}/{N_PEERS}/n_sum.txt', "w") as f:
        for host_data in data:
            target_range = host_data[110+10*XLIM_LEFT:110+10*XLIM_RIGHT]
            f.write(f'{sum(target_range)/80}\n') # total network usage (kB) per host.

    for host_data in data:
        x_values = [(0.1 * i) - 11 for i in range(len(host_data))][:(1-window)]
        plt.plot(
            x_values, 
            np.convolve(host_data, np.ones(window)/window, mode='valid'), 
            color='black', alpha=0.01)
        # x_values = [0.1 * idx for idx in range(len(line_data))]
        # plt.plot(x_values, line_data, color='gray', alpha=0.5)

    plt.ylim(0, 4000)
    plt.xlim(XLIM_LEFT, XLIM_RIGHT)
    plt.xlabel('Time (s)')
    plt.ylabel('Network usage (kb/s)')
    plt.title('Multiple Line Graph')
    plt.grid(True)
    plt.savefig(f'../{RESULTS_PATH}/{N_PEERS}_n_plot.jpg', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# for path in [f'burst/{pkg}' for pkg in ["client-server", "dev-eval-trickle", "dev-v2"]]:
#     RESULTS_PATH = path
#     for n_peers in [130, 160, 190]:
#         N_PEERS = n_peers
#         main()

for path in [f'ablation/dev-v2/t_min_{t_min}_unit_300' for t_min in [0, 200, 400, 600]]:
    RESULTS_PATH = path
    main()

for path in [f'ablation/dev-v2/t_min_300_unit_{t_unit}' for t_unit in [100, 300, 500, 700]]:
    RESULTS_PATH = path
    main()